# Estimation des Stocks de Carbone et de la Biomasse Forestière

## Introduction
La mesure du carbone piégé dans les forêts de la RDC est un enjeu majeur pour le marché des crédits carbone et la lutte contre le réchauffement climatique. Ce notebook utilise des modèles allométriques basés sur l'imagerie satellite pour estimer la biomasse aérienne et le tonnage de carbone stocké à l'hectare.

## Objectifs
*   **Mesure de biomasse** : Convertir le signal végétal en masse ligneuse estimée.
*   **Comptabilité carbone** : Traduire la biomasse en tonnes de carbone (ratio standard de 50%).
*   **Soutien REDD+** : Fournir des données spatialisées pour les projets de conservation forestière.

## Méthodologie
1.  **Setup** : Installation des outils géospatiaux.
2.  **Acquisition** : Images Sentinel-2 (Bandes Rouge et NIR).
3.  **Analyse de densité** : Calcul de l'indice NDVI stable (moyenne annuelle).
4.  **Modélisation** : Application d'une formule allométrique de conversion biomasse/carbone.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib seaborn -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Alignement sur la zone régionale du Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Nous exportons une mosaïque annuelle pour obtenir un indice NDVI stable, non biaisé par la saisonnalité.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B4', 'B8']), 'biomass_input.tif', scale=30, region=roi)

## Modélisation de la Biomasse et du Carbone
L'estimation utilise une relation exponentielle simplifiée : la biomasse est corrélée au carré du NDVI. On considère ensuite que la biomasse ligneuse contient 50% de carbone pur.

In [ ]:
# ====================================================
# ÉTAPE 4 : Calcul Carbone
# ====================================================
with rasterio.open('biomass_input.tif') as src: data = src.read().astype(np.float32)
red, nir = data[0], data[1]
ndvi = (nir - red) / (nir + red + 1e-8)

# Formule simplifiée : Biomass (t/ha) = 150 * NDVI^2
biomass = 150 * np.maximum(0, ndvi)**2
carbon = 0.5 * biomass

plt.figure(figsize=(10, 8))
plt.imshow(carbon, cmap='YlGn')
plt.colorbar(label='Carbone stocké estimé (Tonne/Hectare)')
plt.title('Cartographie des Stocks de Carbone Forestier')
plt.axis('off')
plt.show()